# Modules

In [ ]:
from Utils import Timer, Duur, meet_duur,toon_duur, convert_wkt_string_to_gps, batch_transform_geometry_to_gps_string, convert_coords_to_gps

from scipy.spatial.distance import cdist
from IPython.display import display
import geopandas as gpd
from pyproj import Transformer
from shapely import wkt
from shapely.geometry.base import BaseGeometry
import re
from scipy.spatial import cKDTree
import zipfile
import urllib.request
from pathlib import Path
import pandas as pd
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)       # Geen omvouwing naar volgende regel
pd.set_option('display.float_format', '{:.6f}'.format)  # 6 decimalen voor floats
pd.set_option('display.max_seq_items', None)
pd.set_option('display.precision', 10) 
pd.set_option('display.show_dimensions', True)
import numpy as np
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

# 1 Wegvakken inladen & filteren


In [ ]:
wegvakken_ = gpd.read_file(
    "Data/nwb-wegen-08_01_2026.gpkg",
    layer="wegvakken")
print(wegvakken_.columns)

wegvakken = wegvakken_.copy()
wegvakken = wegvakken[wegvakken['wegnr_hmp'].isin(['A12', 'A4', 'A20', 'N11',
                                                   'N14', 'A13', 'A44', 'A16', 'N44',
                                                   'A27', 'A2'])]
wegvakken = wegvakken.rename(columns={'geometry': 'wegvak_geometry'})
wegvakken = wegvakken[['wvk_id', 'rijrichtng', 'wegnummer', 'wegnr_hmp', 'wegnr_aw',
                    'beginkm', 'eindkm', 'wegbehnaam', 'distrnaam', 'wegvak_geometry']]
wegvakken.to_pickle("Data/wegvakken.pkl")
wegvakken.sample(1).T

Index(['objectid', 'wvk_id', 'wvk_begdat', 'jte_id_beg', 'jte_id_end',
       'wegbehsrt', 'wegnummer', 'wegdeelltr', 'hecto_lttr', 'bst_code',
       'rpe_code', 'admrichtng', 'rijrichtng', 'stt_naam', 'stt_bron',
       'wpsnaam', 'gme_id', 'gme_naam', 'hnrstrlnks', 'hnrstrrhts',
       'e_hnr_lnks', 'e_hnr_rhts', 'l_hnr_lnks', 'l_hnr_rhts', 'begafstand',
       'endafstand', 'beginkm', 'eindkm', 'pos_tv_wol', 'wegbehcode',
       'wegbehnaam', 'distrcode', 'distrnaam', 'dienstcode', 'dienstnaam',
       'wegtype', 'wgtype_oms', 'routeltr', 'routenr', 'routeltr2', 'routenr2',
       'routeltr3', 'routenr3', 'routeltr4', 'routenr4', 'wegnr_aw',
       'wegnr_hmp', 'geobron_id', 'geobron_nm', 'bronjaar', 'openlr',
       'bag_orl', 'frc', 'fow', 'alt_naam', 'alt_nr', 'rel_hoogte',
       'st_lengthshape', 'geometry'],
      dtype='str')

In [ ]:
wegvakken = pd.read_pickle("Data/wegvakken.pkl")
print(f'Wegvakken geladen : {wegvakken.shape[0]:,} rijen, \
{wegvakken.shape[1]} kolommen')
wegvakken.sample(1).T

# 2.3 Hectopunten inladen, schoonmaken & koppelen aan wegvakken

In [ ]:
hectopunten_ = gpd.read_file("Data/nwb-wegen-08_01_2026.gpkg", layer='hectopunten')
hectopunten = hectopunten_.copy()

In [ ]:
print(f'Hectopunten geladen: {hectopunten.shape[0]:,} rijen, {hectopunten.shape[1]} kolommen')
display(hectopunten.sample(1).T)

In [ ]:
hectopunten = hectopunten.rename(columns={
    'geometry': 'hectopunt_geometry',
})
hectopunten = hectopunten.sort_values(by='wvk_id', ascending=True).reset_index()
hectopunten['meter'] = hectopunten['hectomtrng'] * 1000
hectopunten = hectopunten.astype({'wvk_id': 'int',
                                  'hectomtrng': 'int',
                                  'hecto_lttr': 'string',
                                  'hecto_lttr': 'string'})
hectopunten = hectopunten[['hectomtrng', 'afstand', 'wvk_id', 'hecto_lttr', 'hectopunt_geometry']]

In [ ]:
hectopunten.head(1).T

In [ ]:
hectopunten_voor = len(hectopunten)
wegvakken_voor = len(wegvakken)
df = hectopunten.merge(
    wegvakken, 
    on='wvk_id',
    how='inner'
)
df.shape
print(f'Hectopunten voor merge : {hectopunten_voor:,}')
print(f'Wegvakken  voor merge  : {wegvakken_voor:,}')
print(f'Hectopunten na merge   : {df.shape[0]:,}')
print(f'Verlies hectopunten    : {hectopunten_voor - df.shape[0]:,} \
({(hectopunten_voor-df.shape[0])/hectopunten_voor*100:.1f}%)')
print(f'Kolommen na merge ({len(df.columns)}): {df.columns.tolist()}')
display(df.head(1).T)

In [ ]:
df['Zijde'] = np.where(df['beginkm'] < df['eindkm'], 'Rechts', 'Links')
df['oplopend'] = np.where(df['beginkm'] < df['eindkm'], 'R', 'L')
df['snelwegnummer'] = df['wegnr_hmp'].str.replace(r'\D', '', regex=True).astype(int)
df['|'] = ''
df['info'] = ''
df.sample(1).T

# Sloom

In [ ]:
def get_rd_coords_hectopunt(geom):
    if geom is None:
        return None
    if geom.geom_type == 'Point':
        return f"{geom.x:.2f}, {geom.y:.2f}"
    elif geom.geom_type == 'MultiPoint':
        return f"{geom.geoms[0].x:.2f}, {geom.geoms[0].y:.2f}"
    else:
        return None
        
def get_gps_coords_hectopunt(geom):
    if geom is None:
        return None
    transformer = Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True)
    if geom.geom_type == 'Point':
        x, y = geom.x, geom.y
    elif geom.geom_type == 'MultiPoint':
        x, y = geom.geoms[0].x, geom.geoms[0].y
    else:
        return None
    lon, lat = transformer.transform(x, y)
    return f"{lat:.6f}, {lon:.6f}"

def make_streetsmart_link(rd_coord, offset_x=89.35, offset_y=128.84):
    x, y = map(float, rd_coord.split(','))
    minX = round(x - offset_x, 2)
    minY = round(y - offset_y, 2)
    maxX = round(x + offset_x, 2)
    maxY = round(y + offset_y, 2)
    return f"https://streetsmart.cyclomedia.com/streetsmart/?mq={minX};{minY};{maxX};{maxY}&msrs=EPSG:28992"

def make_google_maps_link(gps_coord):
    if gps_coord is None:
        return None
    # De gps_coord is al een string "lat, lon" vanuit je get_gps_coords functie
    return f"https://www.google.com/maps/search/?api=1&query={gps_coord}"

# 1. Genereer eerst de GPS coördinaten (met jouw bestaande functie)
df['gps_coordinaten'] = df['hectopunt_geometry'].apply(get_gps_coords_hectopunt)
# 2. Maak de Google Maps links aan
df['google_maps_link'] = df['gps_coordinaten'].apply(make_google_maps_link)

df['hectopunt_rd_coordinaten'] = df['hectopunt_geometry'].apply(get_rd_coords_hectopunt)
df['streetsmart_link'] = df['hectopunt_rd_coordinaten'].apply(make_streetsmart_link)
df.to_pickle("Output/Data_Analyse-Dataset.pkl")